# Data Exploration with SQLite & DuckDB and Data Preparation for Severe Damage Classification (Case Study: Gorkha District)

1. Distribusi Kasta per Distrik

- import pandas, sqlite3: Memuat library untuk olah data dan koneksi database
- sqlite3.connect('./nepal.sqlite'): Membuka koneksi ke database SQLite
- SELECT ... JOIN id_map ... GROUP BY: Menghitung jumlah rumah tangga per kasta di tiap distrik, lalu mengurutkan dari yang terbanyak
- pd.read_sql_query(query, conn): Menjalankan query dan menyimpan hasilnya ke DataFrame
- conn.close(): Menutup koneksi
- for district in [1,2,3,4] + .head(3): Menampilkan 3 kasta terbesar di tiap distrik

Hasil: Distrik 1, 2 didominasi Chhetree; Distrik 3 didominasi Tamang; Distrik 4 (Gorkha) didominasi Gurung.

In [1]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('./nepal.sqlite')

# Query caste distribution by district
query = """
    SELECT i.district_id, h.caste_household, COUNT(*) as count
    FROM household_demographics h
    JOIN id_map i ON h.household_id = i.household_id
    GROUP BY i.district_id, h.caste_household
    ORDER BY i.district_id, count DESC
"""

df_caste = pd.read_sql_query(query, conn)
conn.close()

# Show top 3 castes per district
for district in [1, 2, 3, 4]:
    print(f"\nDistrict {district}:")
    top_castes = df_caste[df_caste['district_id'] == district].head(3)
    for _, row in top_castes.iterrows():
        print(f"  {row['caste_household']}: {row['count']:,} households")


District 1:
  Chhetree: 8,943 households
  Brahman-Hill: 4,123 households
  Rai: 4,119 households

District 2:
  Chhetree: 15,312 households
  Tamang: 11,927 households
  Newar: 7,451 households

District 3:
  Tamang: 28,402 households
  Brahman-Hill: 19,183 households
  Chhetree: 10,867 households

District 4:
  Gurung: 15,119 households
  Brahman-Hill: 13,043 households
  Chhetree: 8,766 households


2. Instalasi DuckDB
- pip install duckdb → perintah (dikomentari) untuk memasang DuckDB.

In [5]:
# pip install duckdb

3. Import Library
- Memuat pandas, sqlite3, duckdb, numpy, dan time (untuk pengukuran waktu).
- pd.set_option(...) dikomentari; fungsinya menampilkan semua kolom DataFrame.

In [4]:
import pandas as pd
import sqlite3
import duckdb
import numpy as np
import time

# Set display options
# pd.set_option('display.max_columns', None)

4. Melihat Daftar Tabel
- Query sqlite_master (tabel metadata SQLite) untuk mengambil nama semua tabel.
- Hasil: 4 tabel → building_structure, building_damage, household_demographics, id_map.

In [6]:
# 1. Connect to the SQLite database
conn = sqlite3.connect('./nepal.sqlite')

# 2. Query to get all table names
# sqlite_master is a metadata table that contains information about the database schema
tables_df = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)
table_names_list = tables_df['name'].tolist()

# 3. Close the connection
conn.close()

print(f"Tables in database: {table_names_list}")

Tables in database: ['building_structure', 'building_damage', 'household_demographics', 'id_map']


5. Melihat 5 Baris Pertama building_structure
- SELECT * ... LIMIT 5 untuk mengintip struktur kolom (jumlah lantai, usia, luas, tinggi, pondasi, atap, kondisi pasca gempa, dll).

In [7]:
conn = sqlite3.connect('./nepal.sqlite')

df_sqlite_head = pd.read_sql_query("SELECT * FROM building_structure LIMIT 5", conn)

conn.close()
print(df_sqlite_head)

   building_id  count_floors_pre_eq  count_floors_post_eq  age_building  \
0            1                    1                     1             9   
1            2                    1                     1            15   
2            3                    1                     1            20   
3            4                    1                     1            20   
4            5                    1                     1            30   

   plinth_area_sq_ft  height_ft_pre_eq  height_ft_post_eq  \
0                288                 9                  9   
1                364                 9                  9   
2                384                 9                  9   
3                312                 9                  9   
4                308                 9                  9   

  land_surface_condition foundation_type                 roof_type  \
0                   Flat           Other  Bamboo/Timber-Light roof   
1                   Flat           Other  

6. Memilih Kolom Tertentu
- SELECT building_id, age_building → hanya mengambil dua kolom, bukan semua.

In [8]:
conn = sqlite3.connect('./nepal.sqlite')

df_sqlite_columns = pd.read_sql_query("""
    SELECT building_id, age_building
    FROM building_structure
    LIMIT 5
""", conn)

conn.close()
print(df_sqlite_columns)

   building_id  age_building
0            1             9
1            2            15
2            3            20
3            4            20
4            5            30


7. Menghitung Total Bangunan
- COUNT(*) → total 234.835 bangunan.

In [9]:
conn = sqlite3.connect('./nepal.sqlite')

count_df = pd.read_sql_query(
    "SELECT COUNT(*) as count FROM building_structure",
    conn
)
total_buildings_sqlite = count_df.loc[0, 'count']

conn.close()
print(f"Total buildings: {total_buildings_sqlite}")

Total buildings: 234835


8. Jenis Pondasi
- SELECT DISTINCT(foundation_type) → menampilkan nilai unik: Other, Mud mortar-Stone/Brick, Cement-Stone/Brick, Bamboo/Timber, RC.

In [10]:
conn = sqlite3.connect('./nepal.sqlite')

df_foundation_types_sqlite = pd.read_sql_query("""
    SELECT DISTINCT(foundation_type)
    FROM building_structure
""", conn)

conn.close()
print(df_foundation_types_sqlite)

          foundation_type
0                   Other
1  Mud mortar-Stone/Brick
2      Cement-Stone/Brick
3           Bamboo/Timber
4                      RC


9. Data Distrik Gorkha dari SQLite
- WHERE district_id = 4 LIMIT 5 → menyaring tabel id_map untuk Gorkha.

In [11]:
conn = sqlite3.connect('./nepal.sqlite')

df_gorkha_sqlite = pd.read_sql_query("""
    SELECT *
    FROM id_map
    WHERE district_id = 4
    LIMIT 5
""", conn)

conn.close()
print(df_gorkha_sqlite)

   household_id  building_id  vdcmun_id  district_id
0      16400201       164002         38            4
1      16408101       164081         38            4
2      16408901       164089         38            4
3      16409801       164098         38            4
4      16410301       164103         38            4


10. Membaca CSV Langsung dengan DuckDB
- read_csv_auto('./data/building_structure.csv') → DuckDB membaca file CSV langsung tanpa perlu import ke database.
- .fetchdf() → mengubah hasil menjadi DataFrame.

In [12]:
# DuckDB can read CSV files directly!
df_duckdb_head = duckdb.sql("""
    SELECT * FROM read_csv_auto('./data/building_structure.csv') LIMIT 5
""").fetchdf()

print(df_duckdb_head)

   building_id  count_floors_pre_eq  count_floors_post_eq  age_building  \
0            1                    1                     1             9   
1            2                    1                     1            15   
2            3                    1                     1            20   
3            4                    1                     1            20   
4            5                    1                     1            30   

   plinth_area_sq_ft  height_ft_pre_eq  height_ft_post_eq  \
0                288                 9                  9   
1                364                 9                  9   
2                384                 9                  9   
3                312                 9                  9   
4                308                 9                  9   

  land_surface_condition foundation_type                 roof_type  \
0                   Flat           Other  Bamboo/Timber-Light roof   
1                   Flat           Other  

11. Jumlah Bangunan di Gorkha (DuckDB)
- COUNT(*) ... WHERE district_id = 4 → 75.883 bangunan di Gorkha.

In [13]:
count_result = duckdb.sql("""
    SELECT COUNT(*) as count
    FROM read_csv_auto('./data/id_map.csv')
    WHERE district_id = 4
""").fetchdf()

gorkha_count_duckdb = count_result.loc[0, 'count']
print(f"Buildings in Gorkha (District 4): {gorkha_count_duckdb}")

Buildings in Gorkha (District 4): 75883


12. DataFrame Gorkha dengan Index building_id
- SELECT DISTINCT building_id, * EXCLUDE (building_id) → menghilangkan duplikat dan memindahkan building_id ke depan.
- .set_index("building_id") → menjadikannya index. Hasil: shape (75883, 3).

In [14]:
csv_path = './data/id_map.csv'

query = f"""
    SELECT DISTINCT building_id,
           * EXCLUDE (building_id)
    FROM '{csv_path}'
    WHERE district_id = 4
"""

# Convert to DataFrame and set index
df_gorkha_duckdb = duckdb.sql(query).df().set_index("building_id")

print(f"DataFrame shape: {df_gorkha_duckdb.shape}")
print(df_gorkha_duckdb.head())

DataFrame shape: (75883, 3)
             household_id  vdcmun_id  district_id
building_id                                      
164089           16408901         38            4
164103           16410301         38            4
164634           16463401         38            4
164647           16464701         38            4
165374           16537401         30            4


13. Perbandingan Kecepatan SQLite vs DuckDB
- time.time() dipakai untuk mengukur durasi query agregasi (COUNT dan AVG per jenis pondasi).
- Hasil: SQLite 0,558 detik vs DuckDB 0,234 detik → DuckDB sekitar 2,4x lebih cepat.

In [15]:
# 1. SQLite timing - reading from SQLite database
start = time.time()
conn = sqlite3.connect('./nepal.sqlite')
df_sqlite_agg = pd.read_sql_query("""
    SELECT foundation_type, COUNT(*) as count, AVG(age_building) as avg_age
    FROM building_structure
    GROUP BY foundation_type
""", conn)
conn.close()
sqlite_time = time.time() - start

# 2. DuckDB timing - reading from CSV file
start = time.time()
df_duckdb_agg = duckdb.sql("""
    SELECT foundation_type, COUNT(*) as count, AVG(age_building) as avg_age
    FROM read_csv_auto('./data/building_structure.csv')
    GROUP BY foundation_type
""").fetchdf()
duckdb_time = time.time() - start

print(f"SQLite time: {sqlite_time:.4f} seconds")
print(f"DuckDB time: {duckdb_time:.4f} seconds")
if duckdb_time > 0:
    print(f"DuckDB is {sqlite_time/duckdb_time:.1f}x faster!")

SQLite time: 0.5583 seconds
DuckDB time: 0.2341 seconds
DuckDB is 2.4x faster!


14. Fungsi wrangle_nepal_data()

- Parameter csv_path, district_id: Lokasi data dan filter distrik (opsional)
- where_clause: Membuang baris tanpa damage_grade, dan memfilter distrik bila diminta
- Query SELECT DISTINCT ... JOIN: Menggabungkan 3 file: struktur bangunan, tingkat kerusakan, dan id_map
- .set_index("building_id"): Menjadikan ID bangunan sebagai index
- .assign(severe_damage=...): Membuat target biner: 1 jika Grade 4/5 (parah), 0 jika Grade 1–3
- .drop(columns=["damage_grade"]): Menghapus kolom asli agar tidak terjadi data leakage

In [16]:
def wrangle_nepal_data(csv_path="./data", district_id=None):
    """
    Retrieve and prepare Nepal earthquake data for binary classification.

    This function reads CSV files using DuckDB, joins building structure
    with building damage, and creates a binary target variable 'severe_damage' where:
    - 1 = Grade 4 or Grade 5 damage (severe)
    - 0 = Grade 1, 2, or 3 damage (not severe)

    Parameters:
    -----------
    csv_path : str
        Path to the data folder (default: './data')
    district_id : int, optional
        Filter by specific district (e.g., 4 for Gorkha).
        If None, returns all districts.

    Returns:
    --------
    pd.DataFrame
        DataFrame with features and binary target 'severe_damage'
    """
    # Build file paths
    structure_path = f"{csv_path}/building_structure.csv"
    damage_path = f"{csv_path}/building_damage.csv"

    # Build WHERE clause
    where_clause = "WHERE b.damage_grade IS NOT NULL"
    if district_id is not None:
        where_clause += f" AND i.district_id = {district_id}"

    # Always join with id_map to get district_id
    query = f"""
        SELECT DISTINCT
            s.building_id,
            i.district_id,
            s.age_building,
            s.plinth_area_sq_ft,
            s.height_ft_pre_eq,
            s.foundation_type,
            s.ground_floor_type,
            b.damage_grade
        FROM read_csv_auto('{structure_path}') s
        JOIN read_csv_auto('{damage_path}') b ON s.building_id = b.building_id
        JOIN read_csv_auto('./data/id_map.csv') i ON s.building_id = i.building_id
        {where_clause}
    """

    return (
        duckdb.sql(query)
        .df()
        .set_index("building_id")
        .assign(
            severe_damage=lambda x: (
                x["damage_grade"]
                .str.contains("Grade 4|Grade 5")
                .fillna(False)
                .astype(int)
            )
        )
        .drop(columns=["damage_grade"])
    )

15. Pemanggilan Fungsi (dijalankan dua kali, hasil identik)
- wrangle_nepal_data('./data', district_id=4) → data Gorkha siap pakai.
- Hasil: 70.836 baris × 7 kolom, tingkat kerusakan parah 64,26%.

In [17]:
# Gorkha is district_id 4
df_wrangled = wrangle_nepal_data('./data', district_id=4)
print(f"Shape: {df_wrangled.shape}")
print(f"Columns: {list(df_wrangled.columns)}")
print(f"Severe damage rate: {df_wrangled['severe_damage'].mean():.2%}")

Shape: (70836, 7)
Columns: ['district_id', 'age_building', 'plinth_area_sq_ft', 'height_ft_pre_eq', 'foundation_type', 'ground_floor_type', 'severe_damage']
Severe damage rate: 64.26%


16. %%writefile duckdb_wrangle.py
- Menyimpan fungsi wrangling ke file .py agar bisa di-import ulang di notebook lain (from duckdb_wrangle import wrangle_nepal_data).

In [19]:
%%writefile duckdb_wrangle.py
"""
duckdb_wrangle.py

Data wrangling module for Project 4: Nepal Earthquake Damage Prediction.
This module provides functions to query CSV data using DuckDB for machine learning classification tasks.

Usage:
    from duckdb_wrangle import wrangle_nepal_data

    # All data (default)
    df = wrangle_nepal_data("./data")

    # Specific district (e.g., Gorkha = district_id 4)
    df_gorkha = wrangle_nepal_data("./data", district_id=4)
"""

import duckdb
import pandas as pd


def wrangle_nepal_data(csv_path="./data", district_id=None):
    """
    Retrieve and prepare Nepal earthquake data for binary classification.

    This function reads CSV files using DuckDB and joins building structure
    with building damage data. The damage_grade column is kept as-is for use
    in creating the target variable in Lesson 2.

    Parameters:
    -----------
    csv_path : str
        Path to the data folder (default: "./data")
    district_id : int, optional
        Filter by specific district (e.g., 4 for Gorkha).
        If None, returns all districts.

    Returns:
    --------
    pd.DataFrame
        DataFrame with features and damage_grade column
    """
    # Build file paths
    structure_path = f"{csv_path}/building_structure.csv"
    damage_path = f"{csv_path}/building_damage.csv"

    # Build WHERE clause
    where_clause = "WHERE b.damage_grade IS NOT NULL"
    if district_id is not None:
        where_clause += f" AND i.district_id = {district_id}"

    # Always join with id_map to get district_id
    query = f"""
        SELECT DISTINCT
            s.building_id,
            i.district_id,
            s.age_building,
            s.plinth_area_sq_ft,
            s.height_ft_pre_eq,
            s.foundation_type,
            s.ground_floor_type,
            b.damage_grade
        FROM read_csv_auto('{structure_path}') s
        JOIN read_csv_auto('{damage_path}') b ON s.building_id = b.building_id
        JOIN read_csv_auto('./data/id_map.csv') i ON s.building_id = i.building_id
        {where_clause}
    """

    return (
        duckdb.sql(query)
        .df()
        .set_index("building_id")
        .assign(
            severe_damage=lambda x: (
                x["damage_grade"]
                .str.contains("Grade 4|Grade 5")
                .fillna(False)
                .astype(int)
            )
        )
        .drop(columns=["damage_grade"])
    )

Overwriting duckdb_wrangle.py


Kesimpulan Singkat

- Dataset gempa Nepal berisi 234.835 bangunan, terdiri dari 4 tabel yang saling terhubung lewat ID. 
- Distrik Gorkha (pusat gempa) memiliki 75.883 bangunan.
- DuckDB terbukti lebih cepat (±2,4x) daripada SQLite pada query agregasi, serta lebih praktis karena dapat membaca CSV langsung.
- Hasil wrangling: Data Gorkha yang sudah dibersihkan berisi 70836 bangunan dengan 6 fitur (usia, luas lantai, tinggi, jenis pondasi, jenis lantai dasar, distrik) dan target biner severe_damage.
- Sekitar 64% bangunan di Gorkha mengalami kerusakan parah. 
- Kelas target cukup seimbang (tidak ekstrem), sehingga cocok untuk model klasifikasi.
- Data siap dipakai untuk pemodelan Clasification, dengan encoding pada fitur kategorikal seperti foundation_type dan ground_floor_type.